<a href="https://colab.research.google.com/github/akira-kun/SoulX-FlashHead-Colab/blob/main/LTX2_3_T2V_I2V.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: linear-gradient(135deg, #1e1e2f, #2a2a40); padding: 30px; border-radius: 12px; color: white; box-shadow: 0 4px 15px rgba(0,0,0,0.3);">
  <h1 style="margin-top: 0; color: #00e676; letter-spacing: 1px;">🎬 LTX-2.3 Video Generator</h1>
  <p style="font-size: 16px; line-height: 1.6; color: #d1d1e0;">
    Welcome to the streamlined generation engine for <b>Text-to-Video</b> and <b>Image-to-Video</b> using LTX-2.3.
  </p>
  <div style="margin-top: 20px; margin-bottom: 20px;">
    <h3 style="color: #ff5252; margin-bottom: 8px; font-size: 18px;">🔴 <b>Brought to you by <a href="https://youtube.com/@AIWithChucky" target="_blank" style="color: #ff8a80; text-decoration: none; border-bottom: 1px dotted #ff8a80;">AI With Chucky</a></b></h3>
    <p style="font-style: italic; color: #b0bec5; margin-top: 0; font-size: 14px;">Subscribe for more AI tutorials, workflows, and optimization tips!</p>
  </div>
  <hr style="border: 1px solid #4a4a6a; margin: 20px 0;">
  <h3 style="color: #b3b3ff; margin-bottom: 10px;">💠 Instructions:</h3>
  <ol style="font-size: 14px; line-height: 1.8; color: #e6e6f2;">
    <li>Run <b>Cell 1</b> to initialize the environment and download required weights.</li>
    <li>Run <b>Cell 2</b> for pure Text-to-Video generation.</li>
    <li>Run <b>Cell 3</b> to upload an image and animate it.</li>
    <li><b>Important:</b> For width and height, any value you input <b>must be divisible by 32</b> (e.g., 832x480, 512x512) to prevent the server from crashing.</li>
  </ol>
</div>

In [ ]:
# @title ⚙️ System Initialization & Weight Downloads
# @markdown Run this cell once to prepare the environment. The server will remain offline until a generation cell is triggered.

import os
import subprocess
from IPython.display import clear_output, display, HTML
from pathlib import Path

display(HTML("<p style='color: #00e676; font-weight: bold;'>[1/3] Installing core dependencies...</p>"))
!pip install -q torch torchvision torchaudio torchsde einops diffusers accelerate av spandrel albumentations onnx opencv-python onnxruntime tqdm ipywidgets
if not os.path.exists("/content/ComfyUI"):
    !git clone -q https://github.com/comfyanonymous/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt
!apt-get -y install -qq aria2 > /dev/null 2>&1

display(HTML("<p style='color: #00e676; font-weight: bold;'>[2/3] Cloning custom nodes...</p>"))
%cd -q /content/ComfyUI/custom_nodes
nodes = [
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/city96/ComfyUI-GGUF",
    "https://github.com/Lightricks/ComfyUI-LTXVideo/",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",
    "https://github.com/kijai/ComfyUI-MelBandRoFormer"
]
for node in nodes:
    node_name = node.split('/')[-1]
    if not os.path.exists(node_name):
        !git clone -q $node
        req_file = f"{node_name}/requirements.txt"
        if os.path.exists(req_file):
            !pip install -q -r $req_file

display(HTML("<p style='color: #00e676; font-weight: bold;'>[3/3] Fetching LTX-2.3 model weights (this may take a few minutes)...</p>"))
def dl(url, dest, fname):
    Path(dest).mkdir(parents=True, exist_ok=True)
    file_path = os.path.join(dest, fname)
    if not os.path.exists(file_path):
        subprocess.run(['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M', '-d', dest, '-o', fname, url], check=True)

dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-Q4_K_M.gguf", "/content/ComfyUI/models/unet", "ltx-2.3-22b-dev-Q4_K_M.gguf")
dl("https://huggingface.co/Dampfinchen/google-gemma-3-12b-it-qat-q4_0-gguf-small-fix/resolve/main/gemma-3-12b-it-q4_0_s.gguf", "/content/ComfyUI/models/text_encoders", "gemma-3-12b-it-q4_0_s.gguf")
dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/text_encoders/ltx-2.3-22b-dev_embeddings_connectors.safetensors", "/content/ComfyUI/models/text_encoders", "ltx-2.3-22b-dev_embeddings_connectors.safetensors")
dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_video_vae.safetensors", "/content/ComfyUI/models/vae", "ltx-2.3-22b-dev_video_vae.safetensors")
dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_audio_vae.safetensors", "/content/ComfyUI/models/vae", "ltx-2.3-22b-dev_audio_vae.safetensors")
dl("https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.0.safetensors", "/content/ComfyUI/models/latent_upscale_models", "ltx-2.3-spatial-upscaler-x2-1.0.safetensors")
dl("https://huggingface.co/Kijai/MelBandRoFormer_comfy/resolve/main/MelBandRoformer_fp16.safetensors", "/content/ComfyUI/models/diffusion_models", "MelBandRoformer_fp16.safetensors")
dl("https://huggingface.co/Kijai/LTX2.3_comfy/resolve/main/vae/taeltx2_3.safetensors", "/content/ComfyUI/models/vae", "taeltx2_3.safetensors")
dl("https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384.safetensors", "/content/ComfyUI/models/loras", "ltx-2.3-22b-distilled-lora-384.safetensors")

clear_output()
display(HTML("<div style='padding: 15px; background-color: #e8f5e9; border-left: 5px solid #4caf50; border-radius: 4px; color: #2e7d32; font-family: sans-serif;'><b>✨ Initialization Complete!</b> The environment is primed and ready for generation.</div>"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.8/320.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.9/262.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.3/54.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.

In [ ]:
# @title 🎥 Generate Text-to-Video
# @markdown Adjust your parameters below.
positive_prompt = "A cinematic shot of a futuristic sports car driving through a neon-lit cyberpunk city at night. 0-3 seconds: The car speeds past the camera." # @param {type:"string"}
video_width = 832 # @param {type:"integer"}
video_height = 480 # @param {type:"integer"}
video_length_seconds = 5 # @param {type:"slider", min:1, max:10, step:1}
low_vram_mode = True # @param {type:"boolean"}

import json, urllib.request, time, os, glob, socket, subprocess
from PIL import Image
from tqdm.notebook import tqdm
from IPython.display import Video, display, HTML, clear_output

# --- Safety Net: Ensure dimensions are divisible by 32 to prevent crashes ---
safe_width = max(256, round(video_width / 32) * 32)
safe_height = max(256, round(video_height / 32) * 32)
if safe_width != video_width or safe_height != video_height:
    print(f"⚠️ Dimensions auto-adjusted to nearest safe resolution: {safe_width}x{safe_height}")

# --- Server Boot Logic ---
def is_server_running(port=8188):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

if not is_server_running():
    display(HTML("<p style='color: #2196f3; font-family: sans-serif;'>🔄 Booting backend server...</p>"))
    os.chdir('/content/ComfyUI')
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

    cmd = ["python", "main.py", "--dont-print-server"]
    if low_vram_mode:
        cmd.insert(2, "--cache-none")

    subprocess.Popen(cmd)

    while not is_server_running():
        time.sleep(2)
    clear_output()

# --- Workflow Setup ---
workflow_url = 'https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/AICHUCKY-patch-1/Ltx2.3%20.json'
req = urllib.request.Request(workflow_url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as response:
    workflow = json.loads(response.read())

# 1. Create/Overwrite dummy image matching safe dimensions
input_dir = "/content/ComfyUI/input/"
os.makedirs(input_dir, exist_ok=True)
dummy_img = os.path.join(input_dir, "IMG_20210620_151851479_processed.jpg")
Image.new('RGB', (safe_width, safe_height), color='black').save(dummy_img)

# 2. Force Exact Dimensions & FPS
workflow["292"]["inputs"]["value"] = safe_width # Width
workflow["293"]["inputs"]["value"] = safe_height # Height
workflow["285"]["inputs"]["value"] = 12  # FPS

# 3. Force Text-to-Video Mode & User Settings
workflow["290"]["inputs"]["value"] = True
workflow["121"]["inputs"]["text"] = positive_prompt
workflow["291"]["inputs"]["value"] = video_length_seconds

# 4. Force Pass 1 Parameters (Base Generation)
workflow["137"]["inputs"]["sampler_name"] = "lcm"
workflow["360"]["inputs"]["sigmas"] = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
workflow["129"]["inputs"]["cfg"] = 1.0
workflow["115"]["inputs"]["noise_seed"] = 43

# 5. Force Pass 2 Parameters (Upscale/Refine)
workflow["138"]["inputs"]["sampler_name"] = "euler_cfg_pp"
workflow["359"]["inputs"]["sigmas"] = "0.85, 0.7250, 0.4219, 0.0"
workflow["103"]["inputs"]["cfg"] = 1.0
workflow["114"]["inputs"]["noise_seed"] = 42

# --- API Execution & Smart Polling ---
def queue_prompt(pw):
    data = json.dumps({"prompt": pw}).encode('utf-8')
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
    try:
        response = urllib.request.urlopen(req)
        return json.loads(response.read())
    except urllib.error.HTTPError as e:
        error_data = e.read().decode('utf-8')
        print("\n❌ API ERROR:")
        print(error_data)
        raise e

def check_job_status(pid):
    try:
        history = json.loads(urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:8188/history/{pid}")).read())
        if str(pid) in history:
            return "DONE"

        queue = json.loads(urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/queue")).read())
        for job in queue.get('queue_running', []) + queue.get('queue_pending', []):
            if str(job[1]) == str(pid):
                return "RUNNING"

        return "FAILED"
    except Exception:
        return "SERVER_DEAD"

prompt_id = queue_prompt(workflow)['prompt_id']

# --- Timer ---
start_time = time.time()
with tqdm(desc="✨ Rendering Video", bar_format="{desc} | Elapsed Time: {postfix}") as pbar:
    while True:
        elapsed = int(time.time() - start_time)
        mins, secs = divmod(elapsed, 60)
        pbar.set_postfix_str(f"{mins:02d}:{secs:02d}")

        status = check_job_status(prompt_id)

        if status == "DONE":
            pbar.set_description("✅ Render Complete")
            break
        elif status == "FAILED":
            pbar.set_description("❌ Render Failed (Check Memory)")
            break
        elif status == "SERVER_DEAD":
            pbar.set_description("💀 Server Crashed")
            break

        time.sleep(2)
        pbar.update(0)

# --- Output Handling ---
if status == "DONE":
    output_dir = "/content/ComfyUI/output/"
    mp4_files = glob.glob(f"{output_dir}*.mp4") + glob.glob(f"{output_dir}video/*.mp4")
    if mp4_files:
        latest = max(mp4_files, key=os.path.getctime)
        display(HTML("<br>"))
        display(Video(latest, embed=True, width=640))
    else:
        display(HTML("<p style='color: #f44336;'>⚠️ Generation finished, but the video file could not be found.</p>"))

In [4]:
# @title 🖼️ Generate Image-to-Video
# @markdown Running this cell will trigger a prompt to upload your starting image.
positive_prompt = "A young female gamer streamer named Stella, 18 years old, slim, pale skin, short messy blonde hair with bangs, wearing glasses and subtle gothic style makeup. She is sitting in a pink and white gaming chair in a cozy, dimly lit bedroom with purple and pink LED lighting. Behind her there is a neon sign that says \"Stella Byte\" with a heart. She is wearing a black outfit with a choker, and cat-ear headphones glowing softly. She is looking directly at the camera, speaking naturally like a YouTube gaming presenter, with expressive but natural hand gestures. Her fingernails are painted black. The scene is cinematic, soft lighting from a monitor illuminating her face, high detail, ultra realistic, consistent face, stable identity, no distortion, 4K quality, depth of field, natural motion, realistic skin texture. Streamer setup, cozy gamer room, night environment, city lights visible through the window, high quality webcam perspective, front-facing camera angle. Totally muted video, no audio at all, but using light hand gestures to make a point, expressive facial acting." # @param {type:"string"}
video_width = 832 # @param {type:"integer"}
video_height = 480 # @param {type:"integer"}
video_length_seconds = 3 # @param {type:"slider", min:1, max:10, step:1}
low_vram_mode = True # @param {type:"boolean"}

import json, urllib.request, time, os, glob, shutil, socket, subprocess
from tqdm.notebook import tqdm
from google.colab import files
from IPython.display import Video, display, clear_output, HTML

# --- Safety Net: Ensure dimensions are divisible by 32 to prevent crashes ---
safe_width = max(256, round(video_width / 32) * 32)
safe_height = max(256, round(video_height / 32) * 32)
if safe_width != video_width or safe_height != video_height:
    print(f"⚠️ Dimensions auto-adjusted to nearest safe resolution: {safe_width}x{safe_height}")

# --- Image Upload ---
display(HTML("<p style='color: #9c27b0; font-family: sans-serif; font-weight: bold;'>📤 Please upload your starting image below:</p>"))
uploaded = files.upload()

if not uploaded:
    display(HTML("<p style='color: #f44336;'>⚠️ No image uploaded. Operation cancelled.</p>"))
else:
    input_image_name = list(uploaded.keys())[0]

    # FIX: Write the uploaded image directly from memory to the input folder
    input_dir = "/content/ComfyUI/input/"
    os.makedirs(input_dir, exist_ok=True)
    final_image_path = os.path.join(input_dir, input_image_name)

    with open(final_image_path, "wb") as f:
        f.write(uploaded[input_image_name])

    # Clean up the duplicate file auto-saved by Colab in the working directory
    if os.path.exists(input_image_name):
        os.remove(input_image_name)

    clear_output()

    # --- Server Boot Logic ---
    def is_server_running(port=8188):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            return s.connect_ex(('127.0.0.1', port)) == 0

    if not is_server_running():
        display(HTML("<p style='color: #2196f3; font-family: sans-serif;'>🔄 Booting backend server...</p>"))
        os.chdir('/content/ComfyUI')
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        cmd = ["python", "main.py", "--dont-print-server"]
        if low_vram_mode:
            cmd.insert(2, "--cache-none")
        subprocess.Popen(cmd)

        while not is_server_running():
            time.sleep(2)
        clear_output()

    # --- Workflow Setup ---
    workflow_url = 'https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/AICHUCKY-patch-1/Ltx2.3%20.json'
    req = urllib.request.Request(workflow_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response:
        workflow = json.loads(response.read())

    # 1. Force Exact Dimensions & FPS
    workflow["292"]["inputs"]["value"] = safe_width # Width
    workflow["293"]["inputs"]["value"] = safe_height # Height
    workflow["285"]["inputs"]["value"] = 24  # FPS

    # 2. Force Image-to-Video Mode & Settings
    workflow["290"]["inputs"]["value"] = False
    workflow["167"]["inputs"]["image"] = input_image_name
    workflow["121"]["inputs"]["text"] = positive_prompt
    workflow["291"]["inputs"]["value"] = video_length_seconds

    # 3. Force Pass 1 Parameters (Base Generation)
    workflow["137"]["inputs"]["sampler_name"] = "lcm"
    workflow["360"]["inputs"]["sigmas"] = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
    workflow["129"]["inputs"]["cfg"] = 1.0
    workflow["115"]["inputs"]["noise_seed"] = 43

    # 4. Force Pass 2 Parameters (Upscale/Refine)
    workflow["138"]["inputs"]["sampler_name"] = "euler_cfg_pp"
    workflow["359"]["inputs"]["sigmas"] = "0.85, 0.7250, 0.4219, 0.0"
    workflow["103"]["inputs"]["cfg"] = 1.0
    workflow["114"]["inputs"]["noise_seed"] = 42

    # --- API Execution & Smart Polling ---
    def queue_prompt(pw):
        data = json.dumps({"prompt": pw}).encode('utf-8')
        req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
        try:
            response = urllib.request.urlopen(req)
            return json.loads(response.read())
        except urllib.error.HTTPError as e:
            error_data = e.read().decode('utf-8')
            print("\n❌ API ERROR:")
            print(error_data)
            raise e

    def check_job_status(pid):
        try:
            history = json.loads(urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:8188/history/{pid}")).read())
            if str(pid) in history:
                return "DONE"

            queue = json.loads(urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/queue")).read())
            for job in queue.get('queue_running', []) + queue.get('queue_pending', []):
                if str(job[1]) == str(pid):
                    return "RUNNING"

            return "FAILED"
        except Exception:
            return "SERVER_DEAD"

    prompt_id = queue_prompt(workflow)['prompt_id']

    # --- Timer ---
    start_time = time.time()
    with tqdm(desc="✨ Rendering Video", bar_format="{desc} | Elapsed Time: {postfix}") as pbar:
        while True:
            elapsed = int(time.time() - start_time)
            mins, secs = divmod(elapsed, 60)
            pbar.set_postfix_str(f"{mins:02d}:{secs:02d}")

            status = check_job_status(prompt_id)

            if status == "DONE":
                pbar.set_description("✅ Render Complete")
                break
            elif status == "FAILED":
                pbar.set_description("❌ Render Failed (Check Memory)")
                break
            elif status == "SERVER_DEAD":
                pbar.set_description("💀 Server Crashed")
                break

            time.sleep(2)
            pbar.update(0)

    # --- Output Handling ---
    if status == "DONE":
        output_dir = "/content/ComfyUI/output/"
        mp4_files = glob.glob(f"{output_dir}*.mp4") + glob.glob(f"{output_dir}video/*.mp4")
        if mp4_files:
            latest = max(mp4_files, key=os.path.getctime)
            display(HTML("<br>"))
            display(Video(latest, embed=True, width=640))
        else:
            display(HTML("<p style='color: #f44336;'>⚠️ Generation finished, but the video file could not be found.</p>"))

✨ Rendering Video | Elapsed Time: 